# 03 — Análise exploratória (Fase 5)
**Base:** CAT (predição + XAI) · o mesmo notebook roda para BC trocando `BASE`.

Seções: **1. Univariada** (esta) · 2. Bivariada (χ² + Cramér's V) · 3. Informação mútua · 4. Séries temporais e UF · 5. Desbalanceamento dos alvos.

> Se já existe um `03_eda.ipynb`, copie estas células para dentro dele em vez de sobrescrever.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

from inss_acidentes.common.eda_univariada import (
    ConfigEDA, preparar_dataframe, classificar_colunas, executar_univariada,
    frequencias, plotar_pareto, plotar_histograma,
)

BASE = "cat"            # "cat" ou "beneficios"
RAIZ = Path.cwd().resolve().parents[1] if Path.cwd().name == BASE else Path.cwd()
while not (RAIZ / "configs").exists() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent

cfg = ConfigEDA.from_yaml(RAIZ / "configs" / BASE / "eda.yaml", RAIZ)
df = preparar_dataframe(pd.read_parquet(cfg.entrada))
print(f"{len(df):,} linhas × {df.shape[1]} colunas".replace(",", "."))

## 1. Univariada
### 1.1 Classificação das colunas
Confira se está coerente; corrija em `configs/<base>/eda.yaml` (`categoricas`, `numericas`, `datas`, `excluir`, `candidatos_alvo`).

In [ ]:
tipos = classificar_colunas(df, cfg)
{k: v for k, v in tipos.items()}

### 1.2 Execução completa (grava tabelas e figuras em `reports/<base>/`)

In [ ]:
res = executar_univariada(df, cfg)
display(res["resumo"])

### 1.3 Cobertura de Pareto
Quantas categorias são necessárias para cobrir 80% das ocorrências — base para decidir agrupamentos (CID/CBO/CNAE, município).

In [ ]:
display(res["cobertura"].sort_values("n_categorias", ascending=False))

### 1.4 Pareto de uma variável (inline)

In [ ]:
col = tipos["categoricas"][0]        # troque pela variável de interesse
freq = frequencias(df[col])
plotar_pareto(freq, f"Pareto — {col}", cfg.top_n_pareto, cfg.limiar_pareto, df[col].isna().mean())

### 1.5 Numéricas

In [ ]:
display(res["estatisticas"].T)
for c in tipos["numericas"]:
    display(plotar_histograma(df[c], f"Distribuição — {c}", cfg.recortar_percentil))

### 1.6 Leitura e decisões (preencher)
| Variável | Observação | Decisão (entra? agrupar? descartar?) |
|---|---|---|
| | | |